In [ ]:
!pip -q install pandas-gbq pydata-google-auth google-cloud-bigquery

from google.colab import auth
auth.authenticate_user()

import google.auth
from google.cloud import bigquery
import pytz
from datetime import datetime

PROJECT_ID = "project-2427a001-3a24-4028-9b0"
DEST = "crypto_portfolio.bitso_snapshots"  # dataset.tabla
TABLE_FQN = f"{PROJECT_ID}.{DEST}"

# ✅ Agarra credenciales reales de Colab (no metadata server)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
client = bigquery.Client(project=PROJECT_ID, credentials=creds)

# Día local Monterrey
tz = pytz.timezone("America/Monterrey")
today_local = datetime.now(tz).date().isoformat()  # 'YYYY-MM-DD'

# ¿Ya hay registros hoy?
query = f"""
SELECT COUNT(1) AS n
FROM `{TABLE_FQN}`
WHERE DATE(TIMESTAMP(pulled_at_utc), "America/Monterrey") = @d
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("d", "DATE", today_local)]
)

res = list(client.query(query, job_config=job_config).result())
exists = res[0].n > 0

if exists:
    print(f"Hoy ({today_local}) ya existe snapshot en BigQuery. No se inserta nada ✅")
else:
    df_assets.to_gbq(
        destination_table=DEST,
        project_id=PROJECT_ID,
        if_exists="append"
    )
    print(f"Snapshot insertado para {today_local} ✅ filas: {len(df_assets)}")